# Bug Scoring Pipeline —— 基于 Two-Step 架构的多维评分演示

本 Notebook 使用重构后的 `BugDetector` 模块对 `data/dataset_mini/` 下的所有 playbook 进行多维评分。

**设计理念（参见 CLAUDE_TODO/TODO.md）**：
- **Step 1 (LLM Brain)**: `ClaimExtractor` — 从非结构化 agent 输出中提取"原子声明"，过滤寒暄与情绪表达
- **Step 2 (Algorithmic Judge)**: `ClaimScorer` — 用向量算法计算各声明与约束/上下文的契合度

**评分维度**：
| 维度 | 范围 | 方向 | 对应失效模式 |
|------|------|------|------------|
| Support Score | 0–1 | ↑ 好 | FM-2.2/2.3 幻觉/漂移 |
| Norm Score | -1–1 | ↑ 好 | FM-1.1/1.2 规范违背 |
| Repetition Score | 0–1 | ↓ 好 | FM-1.3 重复/死锁 |
| Health Score | 0–1 | ↑ 好 | 综合健康度 |

**注意**：本 Notebook 只展现评分，不判定错误类型或定位错误位置（这些功能预留在 `ErrorLocator` / `ErrorClassifier` 中）。
评审者通过观察分数分布和低分节点，即可人工发现潜在问题。

## 1. 环境初始化与模块导入

In [15]:
import os
import sys
import json
from pathlib import Path
from typing import List, Dict

import numpy as np
import dotenv
dotenv.load_dotenv(override=True)

# Ensure the project root is on sys.path
project_root = Path(os.getcwd()).parent if os.getcwd().endswith('notebooks') else Path(os.getcwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from chatdev.analyzer import (
    BugDetector,
    ClaimExtractor,
    ClaimScorer,
    InteractionScores,
    ScoredPlaybook,
    save_scored_playbook,
)

print(f"Project root: {project_root}")
print("All imports OK")

Project root: d:\Works\code\winter-like-ai\ChatDev
All imports OK


## 2. 自动发现 Playbook

In [16]:
DATASET_DIR = project_root / "data" / "dataset_mini"

def discover_playbooks(dataset_dir: Path) -> List[Dict]:
    """Scan dataset_dir for subdirectories containing playbook.json."""
    entries = []
    for sub in sorted(dataset_dir.iterdir()):
        if not sub.is_dir():
            continue
        playbook_path = sub / "playbook.json"
        api_records_path = sub / "api_records.jsonl"
        if playbook_path.exists():
            entries.append({
                "dir_name": sub.name,
                "playbook_path": str(playbook_path),
                "api_records_path": str(api_records_path) if api_records_path.exists() else None,
            })
    return entries

playbook_entries = discover_playbooks(DATASET_DIR)
print(f"Discovered {len(playbook_entries)} playbooks:")
for i, e in enumerate(playbook_entries):
    has_api = "yes" if e["api_records_path"] else "no"
    print(f"  [{i+1:2d}] {e['dir_name'][:70]}...  api_records={has_api}")

Discovered 12 playbooks:
  [ 1] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914...  api_records=yes
  [ 2] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331215105...  api_records=yes
  [ 3] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331220035...  api_records=yes
  [ 4] CLI_Text_File_Word_Counter_DefaultOrganization_20260331200439...  api_records=yes
  [ 5] CLI_Text_File_Word_Counter_DefaultOrganization_20260331210834...  api_records=yes
  [ 6] CLI_Text_File_Word_Counter_DefaultOrganization_20260331214035...  api_records=yes
  [ 7] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331212438...  api_records=yes
  [ 8] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215600...  api_records=yes
  [ 9] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215959...  api_records=yes
  [10] Simple_CSV_Data_Processor_CLI_DefaultOrganization_20260331211858...  api_records=yes
  [11] Simple_CSV_Data_Processor_CLI_DefaultOrganization_2026033121

## 3. 批量评分（LLM Extraction 模式）

使用 `use_llm_extraction=True`（LLM-based ClaimExtractor），符合 TODO.md 设计：Step 1 由 LLM 提取原子声明。
若需零成本快速扫描，可将 `use_llm_extraction` 设为 `False` 切换为纯文本句子分割模式。

In [ ]:
detector = BugDetector(
    use_llm_extraction=True,
    repetition_window=3,
)

In [ ]:
RESULTS_DIR = project_root / "data" / "scored_playbooks"
os.makedirs(str(RESULTS_DIR), exist_ok=True)

playbook_paths = [e["playbook_path"] for e in playbook_entries]
print(f"Analyzing {len(playbook_paths)} playbooks (this will call LLM + Embedding APIs)...")

all_scored = detector.analyze_playbooks_batch(playbook_paths)

for s in all_scored:
    pb_dir = Path(s.source_playbook_path).parent.name
    out_path = RESULTS_DIR / f"{pb_dir}_scored.json"
    save_scored_playbook(s, str(out_path))
    print(f"  Saved: {out_path.name}")

print(f"\nDone. {len(all_scored)} playbooks scored.")

## 4. 汇总评分表 —— 按 Playbook

颜色编码：绿色=健康 | 黄色=关注 | 红色=警告

In [18]:
def score_color_ansi(score: float, low_is_bad: bool = True) -> str:
    """Return ANSI color code for a score."""
    if low_is_bad:
        if score >= 0.7: return '\033[92m'
        elif score >= 0.4: return '\033[93m'
        else: return '\033[91m'
    else:
        if score <= 0.3: return '\033[92m'
        elif score <= 0.6: return '\033[93m'
        else: return '\033[91m'

RESET = '\033[0m'

task_groups = {}
for s in all_scored:
    parts = Path(s.source_playbook_path).parent.name.rsplit('_DefaultOrganization_', 1)
    task_name = parts[0] if len(parts) == 2 else Path(s.source_playbook_path).parent.name
    task_groups.setdefault(task_name, []).append(s)

print(f"{'Task':<45} {'Run':>3} {'Support':>13} {'Norm':>13} {'Rep':>11} {'Health':>13}")
print(f"{'':<45} {'':>3} {'(FM-2.2/2.3)':>13} {'(FM-1.1/1.2)':>13} {'(FM-1.3)':>11} {'(composite)':>13}")
print("-" * 110)

for task_name, scored_list in task_groups.items():
    for idx, s in enumerate(scored_list):
        c_support = score_color_ansi(s.support_score_mean, low_is_bad=True)
        c_norm = score_color_ansi((s.norm_score_mean + 1) / 2, low_is_bad=True)
        c_rep = score_color_ansi(s.repetition_score_mean, low_is_bad=False)
        c_health = score_color_ansi(s.health_score_mean, low_is_bad=True)
        
        display_name = task_name if idx == 0 else ""
        print(f"{display_name:<45} {idx+1:>3} "
              f"{c_support}{s.support_score_mean:>12.4f}{RESET} "
              f"{c_norm}{s.norm_score_mean:>12.4f}{RESET} "
              f"{c_rep}{s.repetition_score_mean:>10.4f}{RESET} "
              f"{c_health}{s.health_score_mean:>12.4f}{RESET}")
    print()

Task                                          Run       Support          Norm         Rep        Health
                                                   (FM-2.2/2.3)  (FM-1.1/1.2)    (FM-1.3)   (composite)
--------------------------------------------------------------------------------------------------------------
Basic_Network_Ping_Tool_CLI                     1       0.7624      -0.0008     0.5452       0.5781
                                                2       0.7800      -0.0005     0.5648       0.5785
                                                3       0.8080       0.0680     0.4853       0.6241

CLI_Text_File_Word_Counter                      1       0.6962       0.1360     0.4912       0.5951
                                                2       0.6758       0.0630     0.5028       0.5717
                                                3       0.6935       0.0633     0.4943       0.5805

CLI_Unit_Converter_Temperature                  1       0.7446      -0.0081    

## 5. 按角色评分明细

In [19]:
print(f"{'Role':<28} {'Playbook':<40} {'Turns':>5} {'Support(min)':>18} {'Norm(min)':>16} {'Rep(max)':>16} {'Health(mean)':>18}")
print(f"{'':<28} {'':<40} {'':>5} {'(FM-2.2/2.3)':>18} {'(FM-1.1/1.2)':>16} {'(FM-1.3)':>16} {'(composite)':>18}")
print("-" * 155)

for s in all_scored:
    pb_name = Path(s.source_playbook_path).parent.name[:38]
    for role in s.roles:
        rsum = s.per_role_summary.get(role, {})
        if not rsum:
            continue
        c_health = score_color_ansi(rsum.get('health_mean', 0), low_is_bad=True)
        c_support = score_color_ansi(rsum.get('support_min__fm_2_2_2_3', 0), low_is_bad=True)
        c_norm = score_color_ansi((rsum.get('norm_min__fm_1_1_1_2', 0) + 1) / 2, low_is_bad=True)
        c_rep = score_color_ansi(rsum.get('repetition_max__fm_1_3', 0), low_is_bad=False)
        
        print(f"{role:<28} {pb_name:<40} {rsum.get('num_turns', 0):>5} "
              f"{c_support}{rsum.get('support_min__fm_2_2_2_3', 0):>17.4f}{RESET} "
              f"{c_norm}{rsum.get('norm_min__fm_1_1_1_2', 0):>15.4f}{RESET} "
              f"{c_rep}{rsum.get('repetition_max__fm_1_3', 0):>15.4f}{RESET} "
              f"{c_health}{rsum.get('health_mean', 0):>17.4f}{RESET}")
    print()

Role                         Playbook                                 Turns       Support(min)        Norm(min)         Rep(max)       Health(mean)
                                                                                  (FM-2.2/2.3)     (FM-1.1/1.2)         (FM-1.3)        (composite)
-----------------------------------------------------------------------------------------------------------------------------------------------------------
Chief Product Officer        Basic_Network_Ping_Tool_CLI_DefaultOrg       4            0.3879         -0.0071          0.7363            0.5670
Chief Technology Officer     Basic_Network_Ping_Tool_CLI_DefaultOrg       1            1.0000         -0.0095          0.0000            0.8233
Programmer                   Basic_Network_Ping_Tool_CLI_DefaultOrg       8            0.3272         -0.0018          1.0000            0.5184
Code Reviewer                Basic_Network_Ping_Tool_CLI_DefaultOrg       3            1.0000         -0.0177       

## 6. 评分分布可视化（ASCII Histogram）

In [20]:
all_support = []
all_norm = []
all_rep = []
all_health = []

for s in all_scored:
    for ix in s.interactions:
        all_support.append(ix.aggregate_support_score)
        all_norm.append(ix.aggregate_norm_score)
        all_rep.append(ix.repetition_score)
        all_health.append(ix.overall_health_score)

print(f"Total interactions analyzed: {len(all_health)}")

def ascii_histogram(values, bins=10, title="", low_is_bad=True, width=50):
    """Print an ASCII histogram with color coding."""
    counts, edges = np.histogram(values, bins=bins, range=(0, 1))
    max_count = max(counts) if max(counts) > 0 else 1
    
    print(f"\n{title}")
    print(f"{'Range':<12} {'Count':>6}  {'Distribution'}")
    print("-" * (20 + width))
    
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        cnt = counts[i]
        bar_len = int(cnt / max_count * width)
        
        if low_is_bad:
            color = '\033[92m' if lo >= 0.7 else ('\033[93m' if lo >= 0.4 else '\033[91m')
        else:
            color = '\033[92m' if hi <= 0.3 else ('\033[93m' if hi <= 0.6 else '\033[91m')
        
        bar = '█' * bar_len
        print(f"{color}{lo:.2f}-{hi:.2f}{RESET}     {cnt:>6}  {bar}")
    
    mean_val = np.mean(values)
    print(f"\n  Mean: {mean_val:.4f}  |  Min: {np.min(values):.4f}  |  Max: {np.max(values):.4f}")

ascii_histogram(all_support, bins=10, title="Support Score Distribution (FM-2.2/2.3: 0=unsupported, 1=well-grounded)")
ascii_histogram([(n + 1) / 2 for n in all_norm], bins=10,
                title="Norm Score Distribution (FM-1.1/1.2: mapped 0-1, 0=violation, 1=compliant)")
ascii_histogram(all_rep, bins=10, title="Repetition Score Distribution (FM-1.3: 0=novel, 1=identical)",
                low_is_bad=False)
ascii_histogram(all_health, bins=10, title="Overall Health Score Distribution (0=unhealthy, 1=healthy)")

Total interactions analyzed: 170

Support Score Distribution (FM-2.2/2.3: 0=unsupported, 1=well-grounded)
Range         Count  Distribution
----------------------------------------------------------------------
0.00-0.10          0  
0.10-0.20          0  
0.20-0.30          6  ███
0.30-0.40         25  ███████████████
0.40-0.50         22  █████████████
0.50-0.60         10  ██████
0.60-0.70         16  █████████
0.70-0.80          8  ████
0.80-0.90          0  
0.90-1.00         83  ██████████████████████████████████████████████████

  Mean: 0.7353  |  Min: 0.2413  |  Max: 1.0000

Norm Score Distribution (FM-1.1/1.2: mapped 0-1, 0=violation, 1=compliant)
Range         Count  Distribution
----------------------------------------------------------------------
0.00-0.10          0  
0.10-0.20          0  
0.20-0.30          0  
0.30-0.40          0  
0.40-0.50         95  ██████████████████████████████████████████████████
0.50-0.60         65  ██████████████████████████████████
0.60-0.7

## 7. 单 Playbook 逐 Turn 评分时序

In [21]:
if all_scored:
    target = all_scored[0]
    print(f"Playbook: {Path(target.source_playbook_path).parent.name}")
    print(f"Roles: {', '.join(target.roles)}")
    print()
    
    print(f"{'Turn':>4} {'Role':<28} {'Phase':<24} {'Support':>14} {'Norm':>14} {'Rep':>12} {'Health':>14} {'#Claims':>8}")
    print(f"{'':>4} {'':<28} {'':<24} {'(FM-2.2/2.3)':>14} {'(FM-1.1/1.2)':>14} {'(FM-1.3)':>12} {'(composite)':>14} {'':>8}")
    print("-" * 130)
    
    for ix in target.interactions:
        c_health = score_color_ansi(ix.overall_health_score, low_is_bad=True)
        c_support = score_color_ansi(ix.aggregate_support_score, low_is_bad=True)
        c_norm = score_color_ansi((ix.aggregate_norm_score + 1) / 2, low_is_bad=True)
        c_rep = score_color_ansi(ix.repetition_score, low_is_bad=False)
        
        print(f"{ix.playbook_turn:>4} {ix.role:<28} {ix.phase:<24} "
              f"{c_support}{ix.aggregate_support_score:>13.4f}{RESET} "
              f"{c_norm}{ix.aggregate_norm_score:>13.4f}{RESET} "
              f"{c_rep}{ix.repetition_score:>11.4f}{RESET} "
              f"{c_health}{ix.overall_health_score:>13.4f}{RESET} "
              f"{ix.num_claims:>8}")
else:
    print("No scored playbooks available.")

Playbook: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914
Roles: Chief Product Officer, Chief Technology Officer, Programmer, Code Reviewer, Chief Executive Officer

Turn Role                         Phase                           Support           Norm          Rep         Health  #Claims
                                                             (FM-2.2/2.3)   (FM-1.1/1.2)     (FM-1.3)    (composite)         
----------------------------------------------------------------------------------------------------------------------------------
   1 Chief Product Officer        DemandAnalysis                  0.3879        0.0099      0.0000        0.6125        2
   2 Chief Product Officer        DemandAnalysis                  0.5598        0.0102      0.7363        0.4518        2
   3 Chief Product Officer        DemandAnalysis                  1.0000        0.0034      0.3995        0.7057        0
   4 Chief Product Officer        Manual                          0.57

## 8. 低分 Interaction 详情（高风险节点一览）

列出 Health Score 最低的 interaction，便于人工审查。
只标记评分异常的节点，不自动判定错误类型。

In [22]:
all_interactions = []
for s in all_scored:
    pb_name = Path(s.source_playbook_path).parent.name
    for ix in s.interactions:
        all_interactions.append((pb_name, ix))

all_interactions.sort(key=lambda x: x[1].overall_health_score)

TOP_N = 20
print(f"Showing {min(TOP_N, len(all_interactions))} lowest-health interactions:\n")

print(f"{'#':>3} {'Health':>8} {'Support':>10} {'Norm':>10} {'Rep':>10} {'Role':<26} {'Phase':<22} {'Playbook'}")
print(f"{'':>3} {'(comp)':>8} {'(FM-2.2/2.3)':>10} {'(FM-1.1/1.2)':>10} {'(FM-1.3)':>10} {'':<26} {'':<22} {'':0}")
print("-" * 140)

for rank, (pb_name, ix) in enumerate(all_interactions[:TOP_N], 1):
    c_health = score_color_ansi(ix.overall_health_score, low_is_bad=True)
    print(f"{rank:>3} {c_health}{ix.overall_health_score:>8.4f}{RESET} "
          f"{ix.aggregate_support_score:>9.4f} "
          f"{ix.aggregate_norm_score:>9.4f} "
          f"{ix.repetition_score:>9.4f} "
          f"{ix.role:<26} {ix.phase:<22} {pb_name[:50]}")
    
    if ix.prompt_snippet:
        print(f"    Prompt: {ix.prompt_snippet[:120]}...")
    if ix.output_snippet:
        print(f"    Output: {ix.output_snippet[:120]}...")
    print()

Showing 20 lowest-health interactions:

  #   Health    Support       Norm        Rep Role                       Phase                  Playbook
      (comp) (FM-2.2/2.3) (FM-1.1/1.2)   (FM-1.3)                                                   
--------------------------------------------------------------------------------------------------------------------------------------------
  1   0.2991    0.2987    0.0089    0.9399 Programmer                 CodeReviewModification CLI_Text_File_Word_Counter_DefaultOrganization_202
    Prompt: According to the new user's task, our designed product modality, languages and ideas, our developed first-edition source...
    Output: Here are the modified and complete codes following the required format and addressing the comments:

main.py
```python
'...

  2   0.3307    0.4135    0.0107    0.9698 Programmer                 CodeReviewModification CLI_Text_File_Word_Counter_DefaultOrganization_202
    Prompt: According to the new user's task, our de

## 9. 评分相关性矩阵

In [23]:
if all_health:
    support_arr = np.array(all_support)
    norm_arr = np.array([(n + 1) / 2 for n in all_norm])
    novelty_arr = np.array([1.0 - r for r in all_rep])
    health_arr = np.array(all_health)
    
    print("Score Correlation Matrix (all higher = better):")
    print()
    print(f"{'':>22} {'Support':>10} {'Norm':>10} {'Novelty':>10} {'Health':>10}")
    print(f"{'':>22} {'(FM-2.2/2.3)':>10} {'(FM-1.1/1.2)':>10} {'(1-rep FM-1.3)':>10} {'(comp)':>10}")
    print("-" * 65)
    
    labels = ['Support', 'Norm (mapped)', 'Novelty (1-rep)', 'Health']
    arrays = [support_arr, norm_arr, novelty_arr, health_arr]
    
    for li, ai in zip(labels, arrays):
        row = f"{li:>22}"
        for lj, aj in zip(labels, arrays):
            corr = np.corrcoef(ai, aj)[0, 1]
            row += f" {corr:>9.4f}"
        print(row)
    
    print()
    print("health = 0.35*support + 0.35*norm_mapped + 0.30*novelty")
else:
    print("No data for correlation analysis.")

Score Correlation Matrix (all higher = better):

                          Support       Norm    Novelty     Health
                       (FM-2.2/2.3) (FM-1.1/1.2) (1-rep FM-1.3)     (comp)
-----------------------------------------------------------------
               Support    1.0000   -0.0247   -0.2787    0.4454
         Norm (mapped)   -0.0247    1.0000   -0.0700    0.2205
       Novelty (1-rep)   -0.2787   -0.0700    1.0000    0.6860
                Health    0.4454    0.2205    0.6860    1.0000

health = 0.35*support + 0.35*norm_mapped + 0.30*novelty


## 10. 导出汇总报告

In [24]:
def build_summary_report(scored_list):
    """Build a comprehensive summary report from all scored playbooks."""
    total_interactions = sum(s.num_interactions for s in scored_list)
    
    role_aggregates = {}
    for s in scored_list:
        for role, rsum in s.per_role_summary.items():
            if role not in role_aggregates:
                role_aggregates[role] = {
                    "num_playbooks": 0,
                    "total_turns": 0,
                    "health_means": [],
                    "support_mins__fm_2_2_2_3": [],
                    "norm_mins__fm_1_1_1_2": [],
                }
            ra = role_aggregates[role]
            ra["num_playbooks"] += 1
            ra["total_turns"] += rsum.get("num_turns", 0)
            ra["health_means"].append(rsum.get("health_mean", 0))
            ra["support_mins__fm_2_2_2_3"].append(rsum.get("support_min__fm_2_2_2_3", 0))
            ra["norm_mins__fm_1_1_1_2"].append(rsum.get("norm_min__fm_1_1_1_2", 0))
    
    return {
        "meta": {
            "num_playbooks": len(scored_list),
            "total_interactions": total_interactions,
            "extraction_mode": "fast_sentence_split",
        },
        "overall_stats": {
            "support_mean__fm_2_2_2_3": round(float(np.mean([s.support_score_mean for s in scored_list])), 4),
            "norm_mean__fm_1_1_1_2": round(float(np.mean([s.norm_score_mean for s in scored_list])), 4),
            "repetition_mean__fm_1_3": round(float(np.mean([s.repetition_score_mean for s in scored_list])), 4),
            "health_mean": round(float(np.mean([s.health_score_mean for s in scored_list])), 4),
        },
        "per_task": {
            task_name: {
                "num_runs": len(scored_list),
                "avg_health": round(float(np.mean([s.health_score_mean for s in scored_list])), 4),
                "min_health": round(float(np.min([s.health_score_min for s in scored_list])), 4),
            }
            for task_name, scored_list in task_groups.items()
        },
        "per_role": {
            role: {
                "num_playbooks": ra["num_playbooks"],
                "total_turns": ra["total_turns"],
                "avg_health": round(float(np.mean(ra["health_means"])), 4),
                "avg_support_min__fm_2_2_2_3": round(float(np.mean(ra["support_mins__fm_2_2_2_3"])), 4),
                "avg_norm_min__fm_1_1_1_2": round(float(np.mean(ra["norm_mins__fm_1_1_1_2"])), 4),
            }
            for role, ra in role_aggregates.items()
        },
    }

report = build_summary_report(all_scored)

report_path = RESULTS_DIR / "_summary_report.json"
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Summary report saved to: {report_path.resolve()}")
print(f"\nOverall Statistics:")
print(f"  Playbooks analyzed: {report['meta']['num_playbooks']}")
print(f"  Total interactions: {report['meta']['total_interactions']}")
print(f"  Avg Health Score:   {report['overall_stats']['health_mean']:.4f}")
print(f"  Avg Support (FM-2.2/2.3) Score:  {report['overall_stats']['support_mean__fm_2_2_2_3']:.4f}")
print(f"  Avg Norm (FM-1.1/1.2) Score:     {report['overall_stats']['norm_mean__fm_1_1_1_2']:.4f}")
print(f"  Avg Repetition (FM-1.3):         {report['overall_stats']['repetition_mean__fm_1_3']:.4f}")

Summary report saved to: D:\Works\code\winter-like-ai\ChatDev\data\scored_playbooks\_summary_report.json

Overall Statistics:
  Playbooks analyzed: 12
  Total interactions: 170
  Avg Health Score:   0.5889
  Avg Support (FM-2.2/2.3) Score:  0.7346
  Avg Norm (FM-1.1/1.2) Score:     0.0532
  Avg Repetition (FM-1.3):         0.5083


## 11. 原始数据模块 —— 每节点每种故障得分明细

本模块展示每个 interaction（节点）在三种故障维度上的原始评分明细：
- **FM-2.2/2.3 (Support)**：每条 Claim 对历史上下文的 grounded_score
- **FM-1.1/1.2 (Norm)**：每条规则的 pass_score、fail_score、偏差量 Δ
- **FM-1.3 (Repetition)**：与历史输出的最大余弦相似度

In [25]:
# ============================================================================
# 11.1 节点级故障得分汇总表 (Per-Node Fault Score Summary)
# ============================================================================
# 每一行 = 一个 interaction (node)，展示所有故障维度的聚合分和明细数

print("=" * 145)
print("11.1 节点级故障得分汇总表")
print("=" * 145)

# Build global interaction index
global_idx = 0
pb_short_names = {}

print(f"\n{'Node':>5} {'Playbook':<38} {'Role':<26} {'Turn':>4} {'Phase':<22} "
      f"{'Support':>13} {'#C':>3} {'Norm':>13} {'#R':>3} {'Rep':>11} {'Ref':>4} {'Health':>13}")
print(f"{'':>5} {'':<38} {'':<26} {'':>4} {'':<22} "
      f"{'(FM-2.2/2.3)':>13} {'':>3} {'(FM-1.1/1.2)':>13} {'':>3} {'(FM-1.3)':>11} {'':>4} {'(comp)':>13}")
print("-" * 178)

for s in all_scored:
    pb_full = Path(s.source_playbook_path).parent.name
    pb_short = pb_full[:36] if len(pb_full) > 36 else pb_full
    
    for ix in s.interactions:
        c_health = score_color_ansi(ix.overall_health_score, low_is_bad=True)
        c_support = score_color_ansi(ix.aggregate_support_score, low_is_bad=True)
        c_norm = score_color_ansi((ix.aggregate_norm_score + 1) / 2, low_is_bad=True)
        c_rep = score_color_ansi(ix.repetition_score, low_is_bad=False)
        
        print(f"{global_idx:>5} {pb_short:<38} {ix.role:<26} {ix.playbook_turn:>4} {ix.phase:<22} "
              f"{c_support}{ix.aggregate_support_score:>12.4f}{RESET} {ix.num_claims:>3} "
              f"{c_norm}{ix.aggregate_norm_score:>12.4f}{RESET} {len(ix.norm_score_details):>3} "
              f"{c_rep}{ix.repetition_score:>10.4f}{RESET} {ix.repetition_reference_turn:>4} "
              f"{c_health}{ix.overall_health_score:>12.4f}{RESET}")
        global_idx += 1
    print()

print(f"Total: {global_idx} nodes across {len(all_scored)} playbooks")
# Legend
print(f"\n#C = number of Claims extracted  |  #R = number of Rules from prompt")
print(f"Ref = reference turn for repetition match (-1 means no past to compare)")

11.1 节点级故障得分汇总表

 Node Playbook                               Role                       Turn Phase                        Support  #C          Norm  #R         Rep  Ref        Health
                                                                                                     (FM-2.2/2.3)      (FM-1.1/1.2)        (FM-1.3)             (comp)
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    0 Basic_Network_Ping_Tool_CLI_DefaultO   Chief Product Officer         1 DemandAnalysis               0.3879   2       0.0099   2     0.0000   -1       0.6125
    1 Basic_Network_Ping_Tool_CLI_DefaultO   Chief Product Officer         2 DemandAnalysis               0.5598   2       0.0102   1     0.7363    0       0.4518
    2 Basic_Network_Ping_Tool_CLI_DefaultO   Chief Product Officer         3 DemandAnalysis               1.0000   0       0.0034   1     0.3995

In [26]:
# ============================================================================
# 11.2 Claim 级 Support Score 原始数据 (FM-2.2/2.3: Hallucination/Drift)
# ============================================================================
# 每一行 = 一个 Claim + 其最佳匹配上下文句子的支撑度

print("\n" + "=" * 145)
print("11.2 Claim 级 Support Score (FM-2.2/2.3) — 每声明对上下文的支撑度")
print("=" * 145)

sample_idx = 0
sample = all_scored[sample_idx]
pb_sample_name = Path(sample.source_playbook_path).parent.name

print(f"\n--- Detailed View: {pb_sample_name[:70]} ---")
print(f"{'Turn':>4} {'Role':<26} {'C#':>3} {'Claim Text (truncated)':<60} {'Grounded':>10} {'Best Context (truncated)':<50}")
print("-" * 155)

for ix in sample.interactions:
    if not ix.support_score_details:
        print(f"{ix.playbook_turn:>4} {ix.role:<26} {'-':>3} {'(no claims extracted)':<60} {'1.0000':>10} {'(N/A)':<50}")
        continue
    for sd in ix.support_score_details:
        c_score = score_color_ansi(sd.grounded_score, low_is_bad=True)
        claim_short = sd.claim_text[:58] + ".." if len(sd.claim_text) > 60 else sd.claim_text
        ctx_short = sd.best_support_context[:48] + ".." if len(sd.best_support_context) > 50 else sd.best_support_context
        print(f"{ix.playbook_turn:>4} {ix.role:<26} {sd.claim_id:>3} {claim_short:<60} "
              f"{c_score}{sd.grounded_score:>9.4f}{RESET} {ctx_short:<50}")

print(f"\n--- Claim-Level Statistics (all {len(all_scored)} playbooks) ---")
all_claim_scores = []
all_claim_low_count = 0
for s in all_scored:
    for ix in s.interactions:
        for sd in ix.support_score_details:
            all_claim_scores.append(sd.grounded_score)
            if sd.grounded_score < 0.5:
                all_claim_low_count += 1

if all_claim_scores:
    print(f"  Total claims extracted:        {len(all_claim_scores)}")
    print(f"  Mean grounded_score:           {np.mean(all_claim_scores):.4f}")
    print(f"  Min grounded_score:            {np.min(all_claim_scores):.4f}")
    print(f"  Claims with score < 0.5:       {all_claim_low_count} ({100*all_claim_low_count/len(all_claim_scores):.1f}%)")
else:
    print("  No claims found across all playbooks.")


11.2 Claim 级 Support Score (FM-2.2/2.3) — 每声明对上下文的支撑度

--- Detailed View: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914 ---
Turn Role                        C# Claim Text (truncated)                                         Grounded Best Context (truncated)                          
-----------------------------------------------------------------------------------------------------------------------------------------------------------
   1 Chief Product Officer        0 Given the nature of the task, which involves implementing ..    0.4429 As the Chief Product Officer, to satisfy the new..
   1 Chief Product Officer        1 This is because the tool requires functionality to send IC..    0.3879 Application: can implement visualized game, soft..
   2 Chief Product Officer        0 The task requires functionality that is best suited for an..    0.5598 Given the nature of the task, which involves imp..
   2 Chief Product Officer        1 An Application will allow us to i

In [27]:
# ============================================================================
# 11.3 Rule 级 Norm Score 原始数据 (FM-1.1/1.2: Constraint Violation)
# ============================================================================
# 每一行 = 一条规则 + 其 pass_score / fail_score / Δ
# pb_sample_name, sample defined in cell 11.2

print("\n" + "=" * 145)
print("11.3 Rule 级 Norm Score (FM-1.1/1.2) — 每规则的锚点偏差量")
print("=" * 145)

print(f"\n--- Detailed View: {pb_sample_name[:70]} ---")
print(f"{'Turn':>4} {'Role':<26} {'#':>3} {'Rule Text (truncated)':<60} {'Pass':>9} {'Fail':>9} {'Δ(P-F)':>9}")
print("-" * 135)

for ix in sample.interactions:
    if not ix.norm_score_details:
        print(f"{ix.playbook_turn:>4} {ix.role:<26} {'-':>3} {'(no rules extracted from prompt)':<60} "
              f"{'1.0000':>9} {'0.0000':>9} {'1.0000':>9}")
        continue
    for ri, nd in enumerate(ix.norm_score_details):
        c_score = score_color_ansi((nd.score + 1) / 2, low_is_bad=True)
        rule_short = nd.rule[:58] + ".." if len(nd.rule) > 60 else nd.rule
        print(f"{ix.playbook_turn:>4} {ix.role:<26} {ri:>3} {rule_short:<60} "
              f"{nd.pass_score:>8.4f} {nd.fail_score:>8.4f} "
              f"{c_score}{nd.score:>8.4f}{RESET}")

print(f"\n--- Rule-Level Statistics (all {len(all_scored)} playbooks) ---")
all_rule_scores = []
all_rule_neg_count = 0
for s in all_scored:
    for ix in s.interactions:
        for nd in ix.norm_score_details:
            all_rule_scores.append(nd.score)
            if nd.score < 0:
                all_rule_neg_count += 1

if all_rule_scores:
    print(f"  Total rules evaluated:         {len(all_rule_scores)}")
    print(f"  Mean Δ (delta):                {np.mean(all_rule_scores):.4f}")
    print(f"  Min Δ:                         {np.min(all_rule_scores):.4f}")
    print(f"  Max Δ:                         {np.max(all_rule_scores):.4f}")
    print(f"  Rules with Δ < 0 (violation):  {all_rule_neg_count} ({100*all_rule_neg_count/len(all_rule_scores):.1f}%)")
else:
    print("  No rules found across all playbooks.")


11.3 Rule 级 Norm Score (FM-1.1/1.2) — 每规则的锚点偏差量

--- Detailed View: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914 ---
Turn Role                         # Rule Text (truncated)                                             Pass      Fail    Δ(P-F)
---------------------------------------------------------------------------------------------------------------------------------------
   1 Chief Product Officer        0 As the Chief Product Officer, to satisfy the new user's de..   0.4010   0.3911   0.0099
   1 Chief Product Officer        1 Note that we must ONLY discuss the product modality and do..   0.3898   0.3729   0.0169
   2 Chief Product Officer        0 Given the nature of the task, which involves implementing ..   0.7107   0.7006   0.0102
   3 Chief Product Officer        0 I agree with your assessment. The task requires functional..   0.4242   0.4207   0.0034
   4 Chief Product Officer        0 The new user's task, our developed codes and required depe..   0.2164

In [28]:
# ============================================================================
# 11.4 Repetition Score 原始数据 (FM-1.3: Stuck Loop) + 原始数据导出
# ============================================================================

print("\n" + "=" * 145)
print("11.4 Repetition Score (FM-1.3) — 每节点重复度")
print("=" * 145)

rep_data = []
for s in all_scored:
    pb_name = Path(s.source_playbook_path).parent.name[:45]
    for ix in s.interactions:
        rep_data.append((pb_name, ix))

rep_data.sort(key=lambda x: x[1].repetition_score, reverse=True)

print(f"\n{'Rank':>4} {'Rep(FM-1.3)':>13} {'Ref':>4} {'Role':<26} {'Turn':>4} {'Phase':<24} {'Playbook':<46} {'Output Snippet'}")
print("-" * 155)

for rank, (pb_name, ix) in enumerate(rep_data, 1):
    c_rep = score_color_ansi(ix.repetition_score, low_is_bad=False)
    out_snip = ix.output_snippet[:50].replace('\n', ' ') + "..." if ix.output_snippet else "(empty)"
    print(f"{rank:>4} {c_rep}{ix.repetition_score:>12.4f}{RESET} {ix.repetition_reference_turn:>4} "
          f"{ix.role:<26} {ix.playbook_turn:>4} {ix.phase:<24} {pb_name:<46} {out_snip}")

print(f"\n--- Repetition Statistics ---")
rep_array = np.array([r for _, ix in rep_data for r in [ix.repetition_score]])
high_rep = sum(1 for _, ix in rep_data if ix.repetition_score > 0.9)
mid_rep = sum(1 for _, ix in rep_data if 0.5 <= ix.repetition_score <= 0.9)
low_rep = sum(1 for _, ix in rep_data if ix.repetition_score < 0.5)
print(f"  Nodes with Rep > 0.9 (near-identical):   {high_rep} ({100*high_rep/len(rep_data):.1f}%)")
print(f"  Nodes with 0.5 ≤ Rep ≤ 0.9 (similar):    {mid_rep} ({100*mid_rep/len(rep_data):.1f}%)")
print(f"  Nodes with Rep < 0.5 (novel):            {low_rep} ({100*low_rep/len(rep_data):.1f}%)")
print(f"  Mean Repetition:                         {rep_array.mean():.4f}")
print(f"  Max Repetition:                          {rep_array.max():.4f}")
print(f"  Nodes with Rep == 1.0 (exact duplicate):  {sum(1 for _, ix in rep_data if ix.repetition_score >= 0.9999)}")

# ============================================================================
# 11.5 原始数据导出: 保存为结构化 JSON
# ============================================================================
print("\n" + "=" * 145)
print("11.5 原始数据导出")
print("=" * 145)

raw_export = {"nodes": [], "claims": [], "rules": []}

for s in all_scored:
    pb_name = Path(s.source_playbook_path).parent.name
    for ix in s.interactions:
        raw_export["nodes"].append({
            "playbook": pb_name, "role": ix.role, "playbook_turn": ix.playbook_turn,
            "phase": ix.phase, "phase_turn": ix.phase_turn, "node_index": ix.node_index,
            "num_claims": ix.num_claims,
            "aggregate_support_score__fm_2_2_2_3": ix.aggregate_support_score,
            "aggregate_norm_score__fm_1_1_1_2": ix.aggregate_norm_score,
            "repetition_score__fm_1_3": ix.repetition_score,
            "repetition_reference_turn": ix.repetition_reference_turn,
            "overall_health_score": ix.overall_health_score,
        })
        for sd in ix.support_score_details:
            raw_export["claims"].append({
                "playbook": pb_name, "role": ix.role, "playbook_turn": ix.playbook_turn,
                "claim_id": sd.claim_id, "claim_text": sd.claim_text,
                "best_support_context": sd.best_support_context, "grounded_score": sd.grounded_score,
            })
        for nd in ix.norm_score_details:
            raw_export["rules"].append({
                "playbook": pb_name, "role": ix.role, "playbook_turn": ix.playbook_turn,
                "rule": nd.rule, "pass_score": nd.pass_score, "fail_score": nd.fail_score, "delta": nd.score,
            })

raw_path = RESULTS_DIR / "_raw_data.json"
with open(raw_path, 'w', encoding='utf-8') as f:
    json.dump(raw_export, f, ensure_ascii=False, indent=2)

print(f"Raw data exported to: {raw_path.resolve()}")
print(f"  Nodes:  {len(raw_export['nodes'])}")
print(f"  Claims: {len(raw_export['claims'])}")
print(f"  Rules:  {len(raw_export['rules'])}")


11.4 Repetition Score (FM-1.3) — 每节点重复度

Rank   Rep(FM-1.3)  Ref Role                       Turn Phase                    Playbook                                       Output Snippet
-----------------------------------------------------------------------------------------------------------------------------------------------------------
   1       1.0000    2 Programmer                    6 CodeReviewModification   Basic_Network_Ping_Tool_CLI_DefaultOrganizati  main.py ```python ''' This is the main file for th...
   2       1.0000    1 Programmer                    7 CodeReviewModification   Basic_Network_Ping_Tool_CLI_DefaultOrganizati  main.py ```python ''' This is the main file for th...
   3       1.0000    0 Code Reviewer                 2 CodeReviewComment        Basic_Network_Ping_Tool_CLI_DefaultOrganizati  <INFO> Finished...
   4       1.0000    0 Code Reviewer                 3 CodeReviewComment        Basic_Network_Ping_Tool_CLI_DefaultOrganizati  <INFO> Finished...
   5 

## 12. 预留接口说明

以下接口已在 `bug_detector.py` 中预定义，待后续实现：

### `ErrorLocator` — 错误定位
消费 `InteractionScores`，定位哪些 claim 异常、映射到错误类别、提供 turn 级诊断与证据。

### `ErrorClassifier` — 错误分类
消费 `ErrorLocator` 结果，分配 FM 代码 (1.1–3.3)、生成结构化错误报告、支持跨 run 回归检测。

### `ClaimScorer.compute_plan_action_alignment()` — FM-2.6 预留
计算 action 与 plan step 的对齐分数（参数接口已就绪，待上层 planner 输出规范化 plan_steps 后启用）。

当这些接口实现后，本 Notebook 可扩展：
- 自动错误定位结果表格
- 失效模式分布饼图
- 错误严重程度排序